# Day 30 · 端到端推理服务

**配套讲义**: [`days/day-30.md`](../days/day-30.md) ｜ **需要 GPU（云机器）**

写一个 FastAPI 网关：收 base64/URL 图片 + 文本 → 转发 vLLM → 带超时、重试、限流、结构化日志；并发 10 请求不炸，且日志能追溯到单次耗时。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w5.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 起服务

In [ ]:
import subprocess, sys, time
# 在终端里跑更合适：  python -m src.serve.api --port 8080
# notebook 里演示用 Popen（记得最后 terminate）
print("命令：python -m src.serve.api --port 8080")
print("另开终端：curl -s localhost:8080/health")

## 2. 参数校验的边界测试

In [ ]:
import sys; sys.path.insert(0, "..")
from src.serve.api import validate_request

cases = [
    {"messages": [{"role": "user", "content": "hi"}], "images": []},
    {"messages": [], "images": []},
    {"messages": [{"role": "user", "content": "hi"}], "images": ["x"] * 9},
]
for i, c in enumerate(cases, 1):
    try:
        print(f"case{i}:", validate_request(c))
    except Exception as e:
        print(f"case{i}: 拒绝 → {type(e).__name__}: {e}")

## 3. PII 过滤自检

In [ ]:
import sys; sys.path.insert(0, "..")
from src.serve.api import filter_output

dirty = "您的订单 20260920123456 已发货，手机号 13812345678，地址 杭州市西湖区某路 1 号。"
print("原文:", dirty)
print("过滤:", filter_output(dirty))
print("\n→ 客服日志和回流数据都必须过这一层")

## 验收清单

- [ ] `curl` 能打通 `/health` 和 `/v1/chat`
- [ ] 并发 10 请求不崩、不超时，且日志能查到每一单的耗时
- [ ] 图片校验生效（传一张 20 MB 的图会被友好拒绝，不是崩）
- [ ] `progress/weekly-review.md` 的 W5 段已写；进度表 W5 六天 `[x]`，M5 打卡

**卡住了？** 回看 [`days/day-30.md`](../days/day-30.md) 第五节「容易踩的坑」。

> **明天**：`days/day-31.md` —— W6 Agent 周：工具协议